# 03 — R1 Convergence stylistique inter-modèles

**Hypothèse H1** : la diversité stylistique inter-modèles $D_t$ décroît dans le temps.

**Méthode** :
1. Features style Zilinskas (5D) en format long
2. Z-score global
3. $D_t$ = distance moyenne entre centroïdes de modèles par mois
4. OLS : $D_t = \alpha + \beta \cdot t + \epsilon_t$ — H1 si $\beta < 0$ et p < 0.05

**Input** : `data/interim/battles_with_dates.parquet`
**Outputs** :
- `data/processed/diversity_temporal.parquet`
- `paper/figures/R1_convergence.png`

In [ ]:
from pathlib import Path
import sys

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm

from compariawatch.diversity import STYLE_FEATURES, compute_monthly_diversity

DATA_IN  = ROOT / 'data' / 'interim' / 'battles_with_dates.parquet'
DATA_OUT = ROOT / 'data' / 'processed' / 'diversity_temporal.parquet'
FIG_OUT  = ROOT / 'paper' / 'figures' / 'R1_convergence.png'
RANDOM_STATE = 42

print('Imports OK')

In [ ]:
# Cell 2 — Chargement + filtre temporel
df = pd.read_parquet(DATA_IN)

# Battles avec timestamp (exclut reactions sans date votes)
df = df[df['timestamp'].notna()].copy()
# Décisifs uniquement (cohérent avec analyse BT)
df = df[df['winner'].isin(['model_a', 'model_b'])].copy()

print(f'Battles retenues : {len(df):,}')
print(f'Mois             : {df["month"].nunique()}')
print(f'Features style   : {STYLE_FEATURES}')

In [ ]:
# Cell 3 — Diversité mensuelle D_t
div_df = compute_monthly_diversity(df, method='centroid')
print(div_df.to_string())

DATA_OUT.parent.mkdir(parents=True, exist_ok=True)
div_df.to_parquet(DATA_OUT, index=False)
print(f'\nSauvegardé : {DATA_OUT}')

In [ ]:
# Cell 4 — Régression OLS (test H1)
X = sm.add_constant(div_df['month_idx'])
y = div_df['diversity']
model = sm.OLS(y, X).fit()
print(model.summary())

beta = model.params['month_idx']
pval = model.pvalues['month_idx']
h1 = beta < 0 and pval < 0.05
print(f'\nH1 (convergence) : beta={beta:.4f}, p={pval:.4f} → {"ACCEPTÉE" if h1 else "NON REJETÉE (encore)"}')

In [ ]:
# Cell 5 — Figure principale R1
fig, ax = plt.subplots(figsize=(11, 5))

months = div_df['month']
ax.plot(months, div_df['diversity'], 'o-', lw=2, color='steelblue', label='$D_t$ observé')

xp = np.arange(len(div_df))
yp = model.params['const'] + model.params['month_idx'] * xp
ax.plot(months, yp, '--', color='crimson', lw=2,
        label=f'Tendance ($\\beta$={beta:.3f}, p={pval:.3f})')

ax.set_xlabel('Mois')
ax.set_ylabel('Diversité stylistique inter-modèles')
ax.set_title('R1 — Convergence stylistique des LLM sur Compar:IA')
plt.xticks(rotation=45, ha='right')
ax.legend()
plt.tight_layout()
FIG_OUT.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(FIG_OUT, dpi=200, bbox_inches='tight')
plt.show()
print(f'Figure : {FIG_OUT}')